# VGG19 ImageNet inference

This notebook follows the official torchvision pretrained-model pattern: choose a pretrained weights object, use its matching transforms, run inference, and decode the top predictions. The goal is to understand the full VGG19 image-classification path before reusing VGG19 feature layers in later labs.

In [22]:
from torchvision.models import VGG19_Weights, vgg19

# Choose the official pretrained VGG19 ImageNet weights.
# In torchvision, the weights object also carries the matching preprocessing recipe and labels.
weights = VGG19_Weights.DEFAULT

# Build VGG19 with the pretrained weights and switch to inference/evaluation mode.
model = vgg19(weights=weights)
model.eval()

# Get the input recipe and output label names that match these exact weights.
preprocess = weights.transforms()
categories = weights.meta["categories"]

preprocess, categories[:10]

(ImageClassification(
     crop_size=[224]
     resize_size=[256]
     mean=[0.485, 0.456, 0.406]
     std=[0.229, 0.224, 0.225]
     interpolation=InterpolationMode.BILINEAR
 ),
 ['tench',
  'goldfish',
  'great white shark',
  'tiger shark',
  'hammerhead',
  'electric ray',
  'stingray',
  'cock',
  'hen',
  'ostrich'])

## Resolve project paths and choose an image

Notebook paths are relative to the kernel's current working directory, not always to the notebook file. This cell finds the project root and points to one local image under `data/` so later outputs stay organized.

In [23]:
from pathlib import Path

cwd = Path.cwd()
project_root = cwd if (cwd / "pyproject.toml").exists() else cwd.parent

image_path = project_root / "data" / "my_sample_image.jpg"

cwd, project_root, image_path.exists(), image_path

(WindowsPath('C:/Users/giloz/dev/visual-genai-lab/notebooks'),
 WindowsPath('C:/Users/giloz/dev/visual-genai-lab'),
 True,
 WindowsPath('C:/Users/giloz/dev/visual-genai-lab/data/my_sample_image.jpg'))

## Preprocess the image

The torchvision transform converts a normal RGB image into the tensor format VGG19 expects. The key shape change is from a display image size like `(width, height)` to a model input tensor shaped `[channels, height, width]`, usually `[3, 224, 224]` for this VGG19 recipe.

In [24]:
from PIL import Image

image = Image.open(image_path).convert("RGB")

# Apply the official VGG19 transform: resize, crop, convert to tensor, and normalize.
image_tensor = preprocess(image)

image.size, image_tensor.shape

((5712, 4284), torch.Size([3, 224, 224]))

## Run inference and decode top-5 predictions

VGG19 expects a batch shaped `[N, C, H, W]`, so one image needs an added batch dimension. The model returns 1000 raw ImageNet scores; `softmax` turns them into probabilities, and `topk(5)` keeps the five strongest predictions.

In [25]:
import torch

# VGG19 expects a batch shaped [N, C, H, W].
# unsqueeze(0) wraps this one image tensor (C, H, W) into a batch of size 1.
batch = image_tensor.unsqueeze(0)

# inference_mode disables gradient tracking because we are predicting, not training.
with torch.inference_mode():
    scores = model(batch)

probabilities = scores.squeeze(0).softmax(dim=0)

# Keep the five highest-probability class IDs and their probabilities.
top5_probabilities, top5_class_ids = probabilities.topk(5)

# Pair each class ID with its human-readable ImageNet label.
top5_predictions = [
    (categories[class_id], probability)
    for probability, class_id in zip(top5_probabilities, top5_class_ids, strict=False)
]

print(f"batch shape: {batch.shape}")
print(f"scores shape: {scores.shape}")
print(f"top 5 preds: {top5_predictions}")

batch shape: torch.Size([1, 3, 224, 224])
scores shape: torch.Size([1, 1000])
top 5 preds: [('computer keyboard', tensor(0.8362)), ('mouse', tensor(0.0714)), ('space bar', tensor(0.0297)), ('typewriter keyboard', tensor(0.0139)), ('remote control', tensor(0.0087))]
